# DHPS MMGBSA Binding Prediction — Forward Oracle

Ridge, LightGBM, XGBoost, SVR, GNN (AttentiveFP), BiLSTM 모델을 31-seed 평가합니다.  
데이터: `dataset.csv` (columns: `Smiles`, `MMGBSA dG Bind`)

In [ ]:
# --- Cell 1: Install packages ---
!pip install rdkit lightgbm xgboost torch scikit-learn scipy pandas openpyxl transformers -q

In [ ]:
# --- Cell 2: Clone repo & setup ---
!git clone https://github.com/mschongchulshin/dhps-binding-prediction.git
import os
os.chdir('dhps-binding-prediction')
# dataset.csv를 repo 루트에 복사하거나 Google Drive에서 가져오세요
# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/drive/MyDrive/dataset.csv .
!ls -la

## Data Loading

`data_utils.load_data`를 CSV 버전으로 monkey-patch합니다.

In [ ]:
# --- Cell 3: Import & data load (monkey-patch load_data for CSV) ---
import sys, os
import numpy as np
import pandas as pd

# data_utils monkey-patch: load_data를 CSV 버전으로 교체
import data_utils

def load_data_csv(csv_path="dataset.csv"):
    from rdkit import Chem, RDLogger
    RDLogger.DisableLog("rdApp.*")
    df = pd.read_csv(csv_path)
    df = df[["Smiles", "MMGBSA dG Bind"]].dropna()
    df["Smiles"] = df["Smiles"].astype(str).str.strip()
    df["MMGBSA dG Bind"] = pd.to_numeric(df["MMGBSA dG Bind"], errors="coerce")
    df = df.dropna()
    def canonicalize(s):
        mol = Chem.MolFromSmiles(s)
        return Chem.MolToSmiles(mol, canonical=True) if mol else None
    df["canonical_smiles"] = df["Smiles"].apply(canonicalize)
    df = df.dropna(subset=["canonical_smiles"]).reset_index(drop=True)
    print(f"[load_data_csv] Loaded {len(df)} molecules")
    print(f"[load_data_csv] MMGBSA range: {df['MMGBSA dG Bind'].min():.2f} ~ {df['MMGBSA dG Bind'].max():.2f}")
    return df

data_utils.load_data = load_data_csv
from data_utils import split_data, get_rdkit_features

df = load_data_csv("dataset.csv")
df["mol_id"] = df.index
print(df.head())

## Cell 4: Ridge 31-seed Evaluation

Best alpha = 3708.369 (Optuna 결과 하드코딩).

In [ ]:
# --- Cell 4: Ridge 31-seed ---
import json
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import pearsonr, spearmanr

RIDGE_ALPHA = 3708.369222848137

os.makedirs("results", exist_ok=True)

ridge_results = []
for seed in range(31):
    train_df, val_df, test_df = split_data(df, seed=seed)
    X_train = get_rdkit_features(train_df["canonical_smiles"].tolist())
    X_test  = get_rdkit_features(test_df["canonical_smiles"].tolist())
    y_train = train_df["MMGBSA dG Bind"].values
    y_test  = test_df["MMGBSA dG Bind"].values

    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc  = scaler.transform(X_test)

    model = Ridge(alpha=RIDGE_ALPHA)
    model.fit(X_train_sc, y_train)
    preds = model.predict(X_test_sc)

    rmse = float(np.sqrt(mean_squared_error(y_test, preds)))
    r2   = float(r2_score(y_test, preds))
    sp   = float(spearmanr(y_test, preds)[0])
    mae  = float(np.mean(np.abs(y_test - preds)))
    pcc  = float(pearsonr(y_test, preds)[0])

    ridge_results.append({"seed": seed, "RMSE": rmse, "R2": r2, "Spearman": sp, "MAE": mae, "PCC": pcc})
    print(f"Seed {seed:2d}: RMSE={rmse:.4f}  R²={r2:.4f}  ρ={sp:.4f}")

ridge_mean = {k: float(np.mean([r[k] for r in ridge_results])) for k in ["RMSE","R2","Spearman","MAE","PCC"]}
ridge_std  = {k: float(np.std( [r[k] for r in ridge_results])) for k in ["RMSE","R2","Spearman","MAE","PCC"]}
print(f"\n[Ridge] Mean RMSE={ridge_mean['RMSE']:.4f}±{ridge_std['RMSE']:.4f}")
print(f"[Ridge] Mean R²  ={ridge_mean['R2']:.4f}±{ridge_std['R2']:.4f}")
print(f"[Ridge] Mean ρ   ={ridge_mean['Spearman']:.4f}±{ridge_std['Spearman']:.4f}")

out_ridge = {"mean": ridge_mean, "std": ridge_std, "seeds": ridge_results}
with open("results/ridge_31seed_result.json", "w") as f:
    json.dump(out_ridge, f, indent=2)
print("저장: results/ridge_31seed_result.json")

## Cell 5: LightGBM + XGBoost + SVR 31-seed Evaluation

Best params 하드코딩 (Optuna 불필요).

In [ ]:
# --- Cell 5: LightGBM + XGBoost + SVR 31-seed ---
import lightgbm as lgb
from xgboost import XGBRegressor
from sklearn.svm import SVR

LGB_PARAMS = {
    'n_estimators': 300,
    'max_depth': 8,
    'learning_rate': 0.21577860588866943,
    'num_leaves': 241,
    'subsample': 0.6914478144277556,
    'colsample_bytree': 0.48859784786222643,
    'reg_alpha': 0.005955621350112854,
    'reg_lambda': 0.24360262556508194,
    'min_child_samples': 50,
}

XGB_PARAMS = {
    'n_estimators': 500,
    'max_depth': 3,
    'learning_rate': 0.013962489806087993,
    'subsample': 0.6916336862548942,
    'colsample_bytree': 0.9908032126657517,
    'reg_alpha': 0.00018110282582258183,
    'reg_lambda': 0.018939605376130448,
    'min_child_weight': 7,
}

ml_results = {"LightGBM": [], "XGBoost": [], "SVR": []}

for seed in range(31):
    train_df, val_df, test_df = split_data(df, seed=seed)
    X_train = get_rdkit_features(train_df["canonical_smiles"].tolist())
    X_test  = get_rdkit_features(test_df["canonical_smiles"].tolist())
    y_train = train_df["MMGBSA dG Bind"].values
    y_test  = test_df["MMGBSA dG Bind"].values

    scaler = StandardScaler()
    X_tr_sc = scaler.fit_transform(X_train)
    X_te_sc = scaler.transform(X_test)

    # LightGBM
    lgb_model = lgb.LGBMRegressor(**LGB_PARAMS, random_state=seed, verbosity=-1, n_jobs=-1)
    lgb_model.fit(X_train, y_train)
    p = lgb_model.predict(X_test)
    ml_results["LightGBM"].append({
        "seed": seed,
        "RMSE": float(np.sqrt(mean_squared_error(y_test, p))),
        "R2":   float(r2_score(y_test, p)),
        "Spearman": float(spearmanr(y_test, p)[0])
    })

    # XGBoost
    xgb_model = XGBRegressor(**XGB_PARAMS, random_state=seed, verbosity=0, n_jobs=-1)
    xgb_model.fit(X_train, y_train)
    p = xgb_model.predict(X_test)
    ml_results["XGBoost"].append({
        "seed": seed,
        "RMSE": float(np.sqrt(mean_squared_error(y_test, p))),
        "R2":   float(r2_score(y_test, p)),
        "Spearman": float(spearmanr(y_test, p)[0])
    })

    # SVR
    svr = SVR(kernel="rbf", C=10.0, epsilon=0.1)
    svr.fit(X_tr_sc, y_train)
    p = svr.predict(X_te_sc)
    ml_results["SVR"].append({
        "seed": seed,
        "RMSE": float(np.sqrt(mean_squared_error(y_test, p))),
        "R2":   float(r2_score(y_test, p)),
        "Spearman": float(spearmanr(y_test, p)[0])
    })

    print(f"Seed {seed:2d}: "
          f"LGB={ml_results['LightGBM'][-1]['RMSE']:.3f}  "
          f"XGB={ml_results['XGBoost'][-1]['RMSE']:.3f}  "
          f"SVR={ml_results['SVR'][-1]['RMSE']:.3f}")

ml_out = {}
for model_name, runs in ml_results.items():
    m = {k: float(np.mean([r[k] for r in runs])) for k in ["RMSE","R2","Spearman"]}
    s = {k: float(np.std( [r[k] for r in runs])) for k in ["RMSE","R2","Spearman"]}
    ml_out[model_name] = {"mean": m, "std": s, "seeds": runs}
    print(f"{model_name}: RMSE={m['RMSE']:.4f}±{s['RMSE']:.4f}  R²={m['R2']:.4f}±{s['R2']:.4f}  ρ={m['Spearman']:.4f}±{s['Spearman']:.4f}")

with open("results/ml_31seed_result.json", "w") as f:
    json.dump(ml_out, f, indent=2)
print("저장: results/ml_31seed_result.json")

## Cell 6: GNN (AttentiveFP) 31-seed Evaluation

torch-geometric을 별도 설치한 뒤 AttentiveFP 모델을 31-seed 평가합니다.

In [ ]:
# --- Cell 6a: Install torch-geometric (Colab 전용) ---
!pip install torch-geometric -q

In [ ]:
# --- Cell 6b: GNN (AttentiveFP) 31-seed ---
import torch
import torch.nn as nn
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as PyGLoader
from torch_geometric.nn import AttentiveFP
from transformers import get_cosine_schedule_with_warmup
from sklearn.metrics import mean_absolute_error

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[GNN] Device: {DEVICE}")

GNN_PARAMS = {
    'hidden_dim': 256,
    'num_layers': 2,
    'num_timesteps': 3,
    'dropout': 0.17238086369178526,
    'lr': 0.00246834608351206,
    'batch_size': 32,
}

NODE_DIM = 39
EDGE_DIM = 10

def mol_to_graph(smi, y=0.0):
    from rdkit import Chem
    from rdkit.Chem import AllChem
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    frags = Chem.GetMolFrags(mol, asMols=True)
    if len(frags) > 1:
        mol = max(frags, key=lambda m: m.GetNumAtoms())
    atom_features = []
    for atom in mol.GetAtoms():
        common = [1,5,6,7,8,9,14,15,16,17,34,35,53]
        ohe = [int(atom.GetAtomicNum() == a) for a in common]
        f = ohe + [
            atom.GetDegree() / 6.0,
            atom.GetFormalCharge(),
            atom.GetNumImplicitHs() / 4.0,
            int(atom.GetIsAromatic()),
            int(atom.IsInRing()),
        ]
        f = f[:NODE_DIM] + [0.0] * max(0, NODE_DIM - len(f))
        atom_features.append(f)
    if not atom_features:
        return None
    x = torch.tensor(atom_features, dtype=torch.float)
    edge_index, edge_attr = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bt = bond.GetBondTypeAsDouble()
        ring = int(bond.IsInRing())
        conj = int(bond.GetIsConjugated())
        stereo = int(bond.GetStereo())
        ef = [bt/3.0, ring, conj, stereo/5.0,
              int(bt==1), int(bt==1.5), int(bt==2), int(bt==3), int(ring), int(conj)]
        ef = ef[:EDGE_DIM] + [0.0] * max(0, EDGE_DIM - len(ef))
        for src, dst in [(i, j), (j, i)]:
            edge_index.append([src, dst])
            edge_attr.append(ef)
    if not edge_index:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr  = torch.zeros((0, EDGE_DIM), dtype=torch.float)
    else:
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
        edge_attr  = torch.tensor(edge_attr, dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr,
                y=torch.tensor([y], dtype=torch.float))

def make_graphs(df_part):
    graphs = []
    for smi, y in zip(df_part["canonical_smiles"].tolist(), df_part["MMGBSA dG Bind"].tolist()):
        g = mol_to_graph(smi, y)
        if g is not None:
            graphs.append(g)
    return graphs

class GNNRegressor(nn.Module):
    def __init__(self, hidden_dim, num_layers, num_timesteps, dropout):
        super().__init__()
        self.gnn = AttentiveFP(
            in_channels=NODE_DIM, hidden_channels=hidden_dim,
            out_channels=hidden_dim, edge_dim=EDGE_DIM,
            num_layers=num_layers, num_timesteps=num_timesteps, dropout=dropout,
        )
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )
    def forward(self, data):
        x = self.gnn(data.x, data.edge_index, data.edge_attr, data.batch)
        return self.head(x).squeeze(-1)

@torch.no_grad()
def _gnn_eval(model, dl, device):
    model.eval()
    preds, labels = [], []
    for batch in dl:
        batch = batch.to(device)
        preds.extend(model(batch).cpu().numpy().tolist())
        labels.extend(batch.y.cpu().numpy().tolist())
    return np.array(preds), np.array(labels)

def gnn_train_and_eval(train_graphs, val_graphs, test_graphs, params, seed, device):
    torch.manual_seed(seed); np.random.seed(seed)
    train_dl = PyGLoader(train_graphs, batch_size=params["batch_size"], shuffle=True)
    val_dl   = PyGLoader(val_graphs,   batch_size=128, shuffle=False)
    test_dl  = PyGLoader(test_graphs,  batch_size=128, shuffle=False)

    model = GNNRegressor(params["hidden_dim"], params["num_layers"],
                         params["num_timesteps"], params["dropout"]).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=params["lr"], weight_decay=1e-4)
    total_steps = len(train_dl) * 100
    scheduler = get_cosine_schedule_with_warmup(optimizer, int(total_steps * 0.05), total_steps)
    criterion = nn.MSELoss()

    best_rmse, best_state, patience_cnt = float("inf"), None, 0
    PATIENCE = 20

    for epoch in range(100):
        model.train()
        for batch in train_dl:
            batch = batch.to(device)
            optimizer.zero_grad()
            pred = model(batch)
            loss = criterion(pred, batch.y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
        preds_v, labels_v = _gnn_eval(model, val_dl, device)
        rmse = float(np.sqrt(mean_squared_error(labels_v, preds_v)))
        if rmse < best_rmse:
            best_rmse = rmse
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_cnt = 0
        else:
            patience_cnt += 1
            if patience_cnt >= PATIENCE:
                break

    model.load_state_dict(best_state)
    preds_t, labels_t = _gnn_eval(model, test_dl, device)
    rmse = float(np.sqrt(mean_squared_error(labels_t, preds_t)))
    r2   = float(r2_score(labels_t, preds_t))
    rho  = float(spearmanr(labels_t, preds_t)[0])
    pcc  = float(pearsonr(labels_t, preds_t)[0])
    mae  = float(mean_absolute_error(labels_t, preds_t))
    return {"seed": seed, "RMSE": rmse, "MAE": mae, "R2": r2, "PCC": pcc, "Spearman": rho}

gnn_results = []
for seed in range(31):
    print(f"\n[GNN] Seed {seed:2d}...", flush=True)
    train_df_s, val_df_s, test_df_s = split_data(df, seed=seed)
    train_g = make_graphs(train_df_s)
    val_g   = make_graphs(val_df_s)
    test_g  = make_graphs(test_df_s)
    r = gnn_train_and_eval(train_g, val_g, test_g, GNN_PARAMS, seed, DEVICE)
    gnn_results.append(r)
    print(f"  RMSE={r['RMSE']:.4f}  R²={r['R2']:.4f}  ρ={r['Spearman']:.4f}", flush=True)

gnn_mean = {k: float(np.mean([r[k] for r in gnn_results])) for k in ["RMSE","MAE","R2","PCC","Spearman"]}
gnn_std  = {k: float(np.std( [r[k] for r in gnn_results])) for k in ["RMSE","MAE","R2","PCC","Spearman"]}
print(f"\n[GNN] RMSE={gnn_mean['RMSE']:.4f}±{gnn_std['RMSE']:.4f}  R²={gnn_mean['R2']:.4f}±{gnn_std['R2']:.4f}  ρ={gnn_mean['Spearman']:.4f}±{gnn_std['Spearman']:.4f}")

gnn_out = {"mean": gnn_mean, "std": gnn_std, "seeds": gnn_results}
with open("results/gnn_31seed_result.json", "w") as f:
    json.dump(gnn_out, f, indent=2)
print("저장: results/gnn_31seed_result.json")

## Cell 7: BiLSTM 31-seed Evaluation

`results/augmented_full.pkl`이 repo에 포함되어 있어야 합니다.  
`model4_bilstm.py`의 `train_bilstm`, `predict_forward_bilstm`을 사용합니다.

In [ ]:
# --- Cell 7: BiLSTM 31-seed ---
import model4_bilstm as m4

BILSTM_PARAMS = {
    'embed_dim': 128,
    'hidden_dim': 512,
    'n_layers': 1,
    'dropout': 0.2366051379116888,
    'lr': 0.004749019991935565,
    'batch_size': 32,
}

BILSTM_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[BiLSTM] Device: {BILSTM_DEVICE}")

aug_df = pd.read_pickle("results/augmented_full.pkl")
print(f"[BiLSTM] Augmented dataset: {len(aug_df)} rows")

bilstm_results = []
for seed in range(31):
    print(f"\n[BiLSTM] Seed {seed:2d}...", flush=True)
    train_df_s, val_df_s, test_df_s = split_data(df, seed=seed)

    train_ids = set(train_df_s["mol_id"])
    val_ids   = set(val_df_s["mol_id"])
    test_ids  = set(test_df_s["mol_id"])

    train_aug = aug_df[aug_df["mol_id"].isin(train_ids)].reset_index(drop=True)
    val_orig  = aug_df[aug_df["mol_id"].isin(val_ids)   & (~aug_df["is_augmented"])].reset_index(drop=True)
    test_orig = aug_df[aug_df["mol_id"].isin(test_ids)  & (~aug_df["is_augmented"])].reset_index(drop=True)

    model_bl, _ = m4.train_bilstm(
        train_aug, val_orig,
        embed_dim=BILSTM_PARAMS["embed_dim"],
        hidden_dim=BILSTM_PARAMS["hidden_dim"],
        n_layers=BILSTM_PARAMS["n_layers"],
        dropout=BILSTM_PARAMS["dropout"],
        lr=BILSTM_PARAMS["lr"],
        batch_size=BILSTM_PARAMS["batch_size"],
        epochs=10,
        device=BILSTM_DEVICE,
        seed=seed,
    )

    preds  = m4.predict_forward_bilstm(model_bl, test_orig["canonical_smiles"].tolist(), BILSTM_DEVICE)
    labels = test_orig["MMGBSA dG Bind"].values
    mask   = ~np.isnan(preds) & ~np.isnan(labels)
    preds, labels = preds[mask], labels[mask]

    rmse = float(np.sqrt(mean_squared_error(labels, preds)))
    r2   = float(r2_score(labels, preds))
    rho  = float(spearmanr(labels, preds)[0])
    pcc  = float(pearsonr(labels, preds)[0])
    mae  = float(np.mean(np.abs(labels - preds)))

    bilstm_results.append({"seed": seed, "RMSE": rmse, "MAE": mae, "R2": r2, "PCC": pcc, "Spearman": rho})
    print(f"  RMSE={rmse:.4f}  R²={r2:.4f}  ρ={rho:.4f}", flush=True)

bl_mean = {k: float(np.mean([r[k] for r in bilstm_results])) for k in ["RMSE","MAE","R2","PCC","Spearman"]}
bl_std  = {k: float(np.std( [r[k] for r in bilstm_results])) for k in ["RMSE","MAE","R2","PCC","Spearman"]}
print(f"\n[BiLSTM] RMSE={bl_mean['RMSE']:.4f}±{bl_std['RMSE']:.4f}  R²={bl_mean['R2']:.4f}±{bl_std['R2']:.4f}  ρ={bl_mean['Spearman']:.4f}±{bl_std['Spearman']:.4f}")

bl_out = {"mean": bl_mean, "std": bl_std, "seeds": bilstm_results}
with open("results/bilstm_31seed_result.json", "w") as f:
    json.dump(bl_out, f, indent=2)
print("저장: results/bilstm_31seed_result.json")

## Cell 8: Results Summary

전체 모델 mean±std RMSE, Spearman 요약 출력.

In [ ]:
# --- Cell 8: 결과 요약 출력 ---
print(f"{'Model':<12} {'RMSE mean':>12} {'RMSE std':>10} {'Spearman mean':>14} {'Spearman std':>13}")
print("-" * 65)

summary_rows = [
    ("Ridge",    ridge_mean, ridge_std),
    ("LightGBM", ml_out["LightGBM"]["mean"], ml_out["LightGBM"]["std"]),
    ("XGBoost",  ml_out["XGBoost"]["mean"],  ml_out["XGBoost"]["std"]),
    ("SVR",      ml_out["SVR"]["mean"],       ml_out["SVR"]["std"]),
    ("GNN",      gnn_mean,   gnn_std),
    ("BiLSTM",   bl_mean,    bl_std),
]

for name, m, s in summary_rows:
    print(f"{name:<12} {m['RMSE']:>12.4f} {s['RMSE']:>10.4f} {m['Spearman']:>14.4f} {s['Spearman']:>13.4f}")

print("\n완료! 각 결과는 results/ 폴더에 JSON으로 저장됨.")